# Notebook A — Generate Teacher Answers (Gemini 2.5 Pro)

Runs Gemini 2.5 Pro on the same 100 test questions to get teacher answers.
Saves in the same format as student inference files so the judge notebook can treat it uniformly.

**Output:** `data/teacher_inference_100_TESTONLY.json`

Uses Vertex AI with your Google Cloud credits.

In [1]:
# Cell 0: Config
import os, json, time

PROJECT_DIR = r"C:\Users\adishalit1\Desktop\kd_project"
# PROJECT_DIR = os.path.expanduser("~/kd_project")  # Linux

DATA_DIR = os.path.join(PROJECT_DIR, "data")
N_EVAL = 100

# Vertex AI config
GCLOUD_PROJECT = "project-de5f469e-2543-403c-97e"
GCLOUD_LOCATION = "us-central1"
TEACHER_MODEL = "gemini-2.5-pro"

QUESTIONS_FILE = os.path.join(DATA_DIR, "eval_questions_100.json")
OUT_FILE = os.path.join(DATA_DIR, "teacher_inference_100_TESTONLY.json")

with open(QUESTIONS_FILE) as f:
    q_data = json.load(f)

questions = q_data["questions"]
print(f"Loaded {len(questions)} questions")
print(f"Output: {OUT_FILE}")

Loaded 100 questions
Output: C:\Users\adishalit1\Desktop\kd_project\data\teacher_inference_100_TESTONLY.json


In [2]:
# Cell 1: Generate teacher answers
from google import genai
from google.genai import types

client = genai.Client(
    vertexai=True,
    project=GCLOUD_PROJECT,
    location=GCLOUD_LOCATION,
)

# Resume: load existing answers if file exists
if os.path.exists(OUT_FILE):
    with open(OUT_FILE) as f:
        result = json.load(f)
    done_ids = {s["id"] for s in result["samples"] if s.get("outputs", {}).get("teacher", {}).get("answer")}
    print(f"Resuming — already have {len(done_ids)} answers")
else:
    result = {
        "meta": {
            "source": "eval_questions_100.json",
            "n_samples": N_EVAL,
            "teacher_model": TEACHER_MODEL,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        },
        "models": {"teacher": {"path": TEACHER_MODEL}},
        "samples": [{"id": q["id"], "prompt": q["prompt"], "outputs": {}} for q in questions],
    }
    done_ids = set()

total = 0
for sample in result["samples"]:
    if sample["id"] in done_ids:
        continue

    try:
        resp = client.models.generate_content(
            model=TEACHER_MODEL,
            contents=sample["prompt"],
            config=types.GenerateContentConfig(
                temperature=0.0,
                max_output_tokens=2000,
            ),
        )
        answer = resp.text if hasattr(resp, "text") and resp.text else ""
        sample["outputs"]["teacher"] = {"answer": answer}
        total += 1

        # Save every 10 calls
        if total % 10 == 0:
            with open(OUT_FILE, "w", encoding="utf-8") as f:
                json.dump(result, f, ensure_ascii=False, indent=2)
            print(f"  {total}/{len(questions)} done")

    except Exception as e:
        print(f"  ⚠️ Error on {sample['id']}: {repr(e)[:120]}")
        if "429" in repr(e) or "RESOURCE_EXHAUSTED" in repr(e):
            print("  ⏳ Rate limited — waiting 60s...")
            time.sleep(60)

# Final save
with open(OUT_FILE, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"\n✅ Teacher answers saved: {OUT_FILE}")

# Sanity check
n_with = sum(1 for s in result["samples"] if s.get("outputs", {}).get("teacher", {}).get("answer"))
print(f"Answers: {n_with}/{len(result['samples'])}")

  10/100 done
  20/100 done
  30/100 done
  40/100 done
  50/100 done
  60/100 done
  70/100 done
  80/100 done
  90/100 done
  100/100 done

✅ Teacher answers saved: C:\Users\adishalit1\Desktop\kd_project\data\teacher_inference_100_TESTONLY.json
Answers: 100/100
